# 🚀 01. Moby Analytics — Customer Churn Prediction & Factor Analysis (World SOTA Edition)
### ระบบพยากรณ์ความเสี่ยงลูกค้ายกเลิกบริการระดับ State-of-the-Art บน Google Colab

โน้ตบุ๊กนี้ยกระดับสถาปัตยกรรมสู่มาตรฐานระดับโลก:
- **SOTA Dynamic Features (60+ Signals)**: Multi-scale EWMA, 2nd-order Behavior Acceleration, Burstiness (Fano Factor), และ Shannon Entropies
- **Tri-GBDT Core**: การแข่งขันระหว่าง 3 มหาอำนาจ GBDT — **LightGBM**, **XGBoost**, และ **CatBoost**
- **Super Learner (Stacking Meta-Ensemble)**: รวมผลทำนาย Out-of-Fold (OOF) ผ่าน Meta-Classifier
- **Probability Calibration**: ปรับเทียบค่าความน่าจะเป็นด้วย Isotonic / Platt Scaling
- **Optimal Decision Threshold via $F_2$ & Cost-Benefit Curve**: ค้นหาจุดตัดที่สร้างผลตอบแทนสูงสุดต่อธุรกิจ
- **World-Class Diagnostic Suite**: กราฟวิเคราะห์ระดับโลก 6 มิติ (PR/ROC Overlay, Gains, Decile Lift, Reliability Diagram, Cost Curve)
- **SHAP Explainability**: ถอดรหัสปัจจัยความเสี่ยงรายบุคคลด้วย TreeExplainer


## 📦 Step 1: ติดตั้ง Dependencies และตั้งค่า Environment
รันคำสั่งด้านล่างเพื่อติดตั้งไลบรารีที่จำเป็นสำหรับ Machine Learning, Optuna และ SHAP บน Colab


In [ ]:
# ติดตั้งไลบรารีที่จำเป็นบน Google Colab (ระดับ World SOTA)
!pip install -q lightgbm xgboost catboost optuna shap openpyxl scikit-learn matplotlib seaborn


In [ ]:
import os
import sys
import warnings
import datetime
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from typing import Tuple, Dict, Any

import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostClassifier
import optuna
import shap

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.linear_model import LogisticRegression, RidgeClassifier
from sklearn.isotonic import IsotonicRegression
from sklearn.metrics import (
    roc_auc_score, average_precision_score, brier_score_loss,
    f1_score, fbeta_score, precision_recall_curve, roc_curve,
    confusion_matrix, classification_report
)

# ปิดข้อความเตือนที่ไม่จำเป็น
warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
print('✅ Dependencies & SOTA Environment Loaded Successfully!')


## 📂 Step 2: โหลดข้อมูลไฟล์ Excel (8 Sheets)
คุณสามารถเลือกได้ 2 วิธี:
1. **อัปโหลดไฟล์ตรงๆ เข้า Google Colab** (ผ่านปุ่มอัปโหลดด้านซ้าย หรือรัน cell ด้านล่าง)
2. **ใช้ไฟล์จากเครื่อง / Google Drive**



In [ ]:
# ตรวจสอบและระบุ Path ของไฟล์ Excel
EXCEL_PATH = "[1Moby] Data_example for Bangkok university.xlsx"

# ถ้ายังไม่มีไฟล์ ให้เปิดกล่อง Upload สำหรับ Colab
if not os.path.exists(EXCEL_PATH):
    if os.path.exists(f"../data/{EXCEL_PATH}"):
        EXCEL_PATH = f"../data/{EXCEL_PATH}"
    elif os.path.exists(f"data/{EXCEL_PATH}"):
        EXCEL_PATH = f"data/{EXCEL_PATH}"
    else:
        print("กรุณาอัปโหลดไฟล์ Excel:")
        from google.colab import files
        uploaded = files.upload()
        for fn in uploaded.keys():
            if fn.endswith(".xlsx"):
                EXCEL_PATH = fn
                break

print(f"📁 Selected Excel Path: {EXCEL_PATH}")



## 🧹 Step 3: ทำความสะอาดข้อมูล (Data Cleaning & Normalization)
แปลงข้อมูลจาก 8 Sheets (`Users+User_profile`, `Backend_payment`, `SMS/Email usage 6 sheets`) ให้เป็นตารางข้อมูลสะอาดที่พร้อมนำไปคำนวณ Features


In [ ]:
def load_and_clean_moby_excel(excel_path: str):
    print("⏳ Loading sheets from Excel...")
    xls = pd.ExcelFile(excel_path)
    
    # 1. Customers Sheet
    raw_cust = pd.read_excel(xls, sheet_name="Users+User_profile")
    raw_cust.columns = raw_cust.columns.str.strip()
    
    credit_sms_col = "user.credit + user.credit_premium" if "user.credit + user.credit_premium" in raw_cust.columns else "credit_sms"
    customers = pd.DataFrame()
    customers["acc_id"] = pd.to_numeric(raw_cust["acc_id"], errors="coerce").astype("Int64")
    customers = customers.dropna(subset=["acc_id"]).copy()
    customers["acc_id"] = customers["acc_id"].astype(int)
    customers["status_sms"] = raw_cust.get("status (SMS)", raw_cust.get("status_sms", "")).astype(str).str.strip().str.lower()
    customers["status_email"] = raw_cust.get("status (Email)", raw_cust.get("status_email", "")).astype(str).str.strip().str.lower()
    customers["credit_sms"] = pd.to_numeric(raw_cust[credit_sms_col], errors="coerce").fillna(0.0)
    customers["credit_email"] = pd.to_numeric(raw_cust.get("credit_email", 0), errors="coerce").fillna(0.0)
    
    for dt_col in ["expire", "expire_email", "join_date", "last_access", "last_send"]:
        if dt_col in raw_cust.columns:
            target_name = "expire_sms" if dt_col == "expire" else dt_col
            customers[target_name] = pd.to_datetime(raw_cust[dt_col], errors="coerce", utc=True).dt.tz_localize(None)
            
    # 2. Payments Sheet
    raw_pay = pd.read_excel(xls, sheet_name="Backend_payment")
    raw_pay.columns = raw_pay.columns.str.strip()
    payments = pd.DataFrame()
    payments["acc_id"] = pd.to_numeric(raw_pay["acc_id"], errors="coerce").astype("Int64")
    payments["payment_uid"] = pd.to_numeric(raw_pay.get("uid", raw_pay.get("payment_uid", np.nan)), errors="coerce").astype("Int64")
    payments["payment_date"] = pd.to_datetime(raw_pay["payment_date"], errors="coerce", utc=True).dt.tz_localize(None)
    payments["amount"] = pd.to_numeric(raw_pay["amount"], errors="coerce").fillna(0.0)
    payments["credit_add"] = pd.to_numeric(raw_pay.get("credit_add", 0), errors="coerce").fillna(0.0)
    payments["credit_type"] = raw_pay.get("credit_type", "").astype(str).str.strip().str.lower()
    payments = payments.dropna(subset=["acc_id", "payment_date"]).copy()
    payments["acc_id"] = payments["acc_id"].astype(int)
    payments = payments.sort_values(["acc_id", "payment_date"]).reset_index(drop=True)
    
    # 3. Usage Sheets (6 sheets)
    usage_map = {
        "SMS_usage (BC)": ("sms", "bc"), "SMS_usage (API)": ("sms", "api"), "SMS_usage (OTP)": ("sms", "otp"),
        "Email_usage (BC)": ("email", "bc"), "Email_usage (API)": ("email", "api"), "Email_usage (OTP)": ("email", "otp")
    }
    usage_parts = []
    for sheet_name, (chan, src) in usage_map.items():
        if sheet_name in xls.sheet_names:
            u_df = pd.read_excel(xls, sheet_name=sheet_name)
            u_df.columns = u_df.columns.str.strip()
            if "acc_id" in u_df.columns and "year" in u_df.columns and "month" in u_df.columns:
                part = pd.DataFrame()
                part["acc_id"] = pd.to_numeric(u_df["acc_id"], errors="coerce").astype("Int64")
                part["year"] = pd.to_numeric(u_df["year"], errors="coerce").astype("Int64")
                part["month"] = pd.to_numeric(u_df["month"], errors="coerce").astype("Int64")
                part["usage"] = pd.to_numeric(u_df.get("usage", 0), errors="coerce").fillna(0.0)
                part["channel"] = chan
                part["usage_source"] = src
                part = part.dropna(subset=["acc_id", "year", "month"]).copy()
                part["acc_id"] = part["acc_id"].astype(int)
                part["year"] = part["year"].astype(int)
                part["month"] = part["month"].astype(int)
                usage_parts.append(part)
                
    usage = pd.concat(usage_parts, ignore_index=True)
    usage["period"] = pd.to_datetime(usage["year"].astype(str) + "-" + usage["month"].astype(str).str.zfill(2) + "-01")
    usage = usage.sort_values(["acc_id", "period"]).reset_index(drop=True)
    
    print(f"✅ Loaded: {len(customers):,} customers | {len(payments):,} payments | {len(usage):,} usage records")
    return customers, payments, usage

customers, payments, usage = load_and_clean_moby_excel(EXCEL_PATH)



## ⚙️ Step 4: สกัด Point-in-Time Features และสร้าง Churn Label (World SOTA Feature Set)
- **Cutoff Date**: เส้นแบ่งเวลา "ปัจจุบันจำลอง" (ค่าตั้งต้น: `2025-07-01`)
- **Point-in-Time Safety**: สกัดข้อมูลก่อน Cutoff เท่านั้น ป้องกัน Data Leakage 100%
- **SOTA Dynamic Features**: เพิ่มมิติความเร็ว (Velocity), ความเร่ง (Acceleration), Multi-scale EWMA, ความกระชาก (Fano factor), และ Shannon Entropies


In [ ]:
# กำหนดวัน Cutoff และกรอบเวลาทำนาย
CUTOFF_DATE = "2025-07-01"
HORIZON_DAYS = 180

def extract_churn_sota_features_and_labels(customers, payments, usage, cutoff_str="2025-07-01", horizon_days=180):
    cutoff = pd.Timestamp(cutoff_str)
    horizon_end = cutoff + pd.Timedelta(days=horizon_days)
    w180_start = cutoff - pd.Timedelta(days=180)
    w90_start = cutoff - pd.Timedelta(days=90)
    w30_start = cutoff - pd.Timedelta(days=30)
    
    # ข้อมูลย้อนหลังก่อน Cutoff เท่านั้น (Point-in-Time Safety)
    hist_pay = payments[payments["payment_date"] < cutoff].copy()
    hist_use = usage[usage["period"] < cutoff].copy()
    
    all_ids = sorted(set(customers["acc_id"]) | set(hist_pay["acc_id"]) | set(hist_use["acc_id"]))
    feats = pd.DataFrame({"acc_id": all_ids})
    
    # 1. Profile Features
    profile = customers[["acc_id", "join_date"]].copy()
    profile["customer_age_days"] = (cutoff - profile["join_date"]).dt.days
    feats = feats.merge(profile[["acc_id", "customer_age_days"]], on="acc_id", how="left")
    
    # 2. Payment Features (Standard + SOTA Multi-Window)
    pay_grp = hist_pay.groupby("acc_id")
    p_last = pay_grp["payment_date"].max()
    p_cnt_all = pay_grp["payment_date"].count()
    p_rev_all = pay_grp["amount"].sum()
    p_max_amount = pay_grp["amount"].max()
    
    p_180 = hist_pay[hist_pay["payment_date"] >= w180_start]
    p_cnt_180 = p_180.groupby("acc_id")["payment_date"].count()
    p_rev_180 = p_180.groupby("acc_id")["amount"].sum()
    
    p_90 = hist_pay[hist_pay["payment_date"] >= w90_start]
    p_cnt_90 = p_90.groupby("acc_id")["payment_date"].count()
    p_prev_90 = hist_pay[(hist_pay["payment_date"] >= w180_start) & (hist_pay["payment_date"] < w90_start)]
    p_cnt_prev_90 = p_prev_90.groupby("acc_id")["payment_date"].count()
    
    p_30 = hist_pay[hist_pay["payment_date"] >= w30_start]
    p_cnt_30 = p_30.groupby("acc_id")["payment_date"].count()
    p_rev_30 = p_30.groupby("acc_id")["amount"].sum()
    
    def _intv_mean(s):
        if len(s) < 2: return np.nan
        return float(s.sort_values().diff().dropna().dt.total_seconds().mean() / 86400.0)
    def _cv(s):
        if len(s) < 2: return np.nan
        m = s.mean()
        return float(s.std(ddof=1) / m) if m > 0 else np.nan
        
    p_intv = pay_grp["payment_date"].agg(_intv_mean)
    p_cv = pay_grp["amount"].agg(_cv)
    
    feats["days_since_last_payment"] = feats["acc_id"].map((cutoff - p_last).dt.days)
    feats["payment_count_all"] = feats["acc_id"].map(p_cnt_all).fillna(0).astype(int)
    feats["payment_count_180d"] = feats["acc_id"].map(p_cnt_180).fillna(0).astype(int)
    feats["payment_count_30d"] = feats["acc_id"].map(p_cnt_30).fillna(0).astype(int)
    feats["total_revenue_all"] = feats["acc_id"].map(p_rev_all).fillna(0.0)
    feats["total_revenue_180d"] = feats["acc_id"].map(p_rev_180).fillna(0.0)
    feats["payment_amount_30d"] = feats["acc_id"].map(p_rev_30).fillna(0.0)
    feats["avg_transaction_value"] = np.where(feats["payment_count_all"] > 0, feats["total_revenue_all"] / feats["payment_count_all"], np.nan)
    feats["payment_interval_mean_days"] = feats["acc_id"].map(p_intv)
    feats["payment_overdue_ratio"] = feats["days_since_last_payment"] / feats["payment_interval_mean_days"]
    feats["payment_amount_cv"] = feats["acc_id"].map(p_cv)
    
    c90 = feats["acc_id"].map(p_cnt_90).fillna(0).astype(float)
    cp90 = feats["acc_id"].map(p_cnt_prev_90).fillna(0).astype(float)
    diff_p90 = c90 - cp90
    feats["payment_velocity_ratio"] = np.sign(diff_p90) * np.log1p(np.abs(diff_p90 / (cp90 + 1e-9)))
    p_max = feats["acc_id"].map(p_max_amount).fillna(0.0)
    feats["payment_amount_max_ratio"] = p_max / (feats["avg_transaction_value"] + 1e-9)
    feats["recency_monetary_ratio"] = feats["days_since_last_payment"] / (np.log1p(feats["total_revenue_all"]) + 1e-9)
    
    # 3. Usage Features (Dynamics, Acceleration, EWMA, Burstiness)
    use_pos = hist_use[hist_use["usage"] > 0]
    u_last = use_pos.groupby("acc_id")["period"].max()
    u_tot_180 = hist_use[hist_use["period"] >= w180_start].groupby("acc_id")["usage"].sum()
    u_rec_90 = hist_use[hist_use["period"] >= w90_start].groupby("acc_id")["usage"].sum()
    u_prev_90 = hist_use[(hist_use["period"] >= w180_start) & (hist_use["period"] < w90_start)].groupby("acc_id")["usage"].sum()
    u_rec_30 = hist_use[hist_use["period"] >= w30_start].groupby("acc_id")["usage"].sum()
    u_months_180 = use_pos[use_pos["period"] >= w180_start].groupby("acc_id")["period"].nunique()
    
    monthly_u = hist_use[hist_use["period"] >= w180_start].groupby(["acc_id", "period"])["usage"].sum().unstack(fill_value=0.0)
    
    def _calc_dynamics(row):
        if len(row) <= 1 or row.sum() == 0:
            return pd.Series([0.0, 0.0, 0.0, 0.0, 0.0, 0.0], index=["slope", "velocity", "accel", "fano", "ewma30", "ewma90"])
        x = np.arange(len(row))
        y = row.values.astype(float)
        slope, _ = np.polyfit(x, y, 1)
        v = float(y[-1] - y[-2]) if len(y) >= 2 else 0.0
        a = float((y[-1] - y[-2]) - (y[-2] - y[-3])) if len(y) >= 3 else 0.0
        mean_u = float(np.mean(y))
        var_u = float(np.var(y))
        fano = float(var_u / (mean_u + 1e-9))
        w30 = np.array([0.5**(len(y) - 1 - i) for i in range(len(y))])
        ewma30 = float(np.sum(y * w30) / np.sum(w30))
        w90 = np.array([0.5**((len(y) - 1 - i) / 3.0) for i in range(len(y))])
        ewma90 = float(np.sum(y * w90) / np.sum(w90))
        return pd.Series([float(slope), v, a, fano, ewma30, ewma90], index=["slope", "velocity", "accel", "fano", "ewma30", "ewma90"])
        
    u_dyn = monthly_u.apply(_calc_dynamics, axis=1) if not monthly_u.empty else pd.DataFrame(columns=["slope", "velocity", "accel", "fano", "ewma30", "ewma90"])
    
    feats["days_since_last_usage"] = feats["acc_id"].map((cutoff - u_last).dt.days)
    feats["usage_total_180d"] = feats["acc_id"].map(u_tot_180).fillna(0.0)
    feats["usage_recent_90d"] = feats["acc_id"].map(u_rec_90).fillna(0.0)
    feats["usage_prev_90d"] = feats["acc_id"].map(u_prev_90).fillna(0.0)
    feats["usage_recent_30d"] = feats["acc_id"].map(u_rec_30).fillna(0.0)
    feats["usage_30d_to_90d_ratio"] = feats["usage_recent_30d"] / (feats["usage_recent_90d"] + 1e-9)
    feats["usage_active_months_180d"] = feats["acc_id"].map(u_months_180).fillna(0).astype(int)
    feats["usage_consistency_ratio"] = feats["usage_active_months_180d"] / 6.0
    feats["usage_slope_6m"] = feats["acc_id"].map(u_dyn["slope"]).fillna(0.0)
    feats["usage_velocity"] = feats["acc_id"].map(u_dyn["velocity"]).fillna(0.0)
    feats["usage_acceleration"] = feats["acc_id"].map(u_dyn["accel"]).fillna(0.0)
    feats["usage_burstiness_fano"] = feats["acc_id"].map(u_dyn["fano"]).fillna(0.0)
    feats["usage_ewma_30d"] = feats["acc_id"].map(u_dyn["ewma30"]).fillna(0.0)
    feats["usage_ewma_90d"] = feats["acc_id"].map(u_dyn["ewma90"]).fillna(0.0)
    
    diff_u = feats["usage_recent_90d"] - feats["usage_prev_90d"]
    feats["usage_change_90d_pct"] = np.sign(diff_u) * np.log1p(np.abs(diff_u / (feats["usage_prev_90d"] + 1e-9)))
    ratio_u = feats["usage_recent_90d"] / (feats["usage_prev_90d"] + 1e-9)
    feats["usage_decay_ratio"] = np.sign(ratio_u) * np.log1p(np.abs(ratio_u))
    
    # 4. Channel Diversity & Shannon Entropies
    u_chan = hist_use.groupby(["acc_id", "channel"])["usage"].sum().unstack(fill_value=0.0)
    u_src = hist_use.groupby(["acc_id", "usage_source"])["usage"].sum().unstack(fill_value=0.0)
    tot_u = hist_use.groupby("acc_id")["usage"].sum()
    t_map = feats["acc_id"].map(tot_u).fillna(0.0)
    
    feats["sms_usage_share"] = np.where(t_map > 0, feats["acc_id"].map(u_chan.get("sms", 0)).fillna(0.0) / t_map, 0.0)
    feats["email_usage_share"] = np.where(t_map > 0, feats["acc_id"].map(u_chan.get("email", 0)).fillna(0.0) / t_map, 0.0)
    feats["bc_usage_share"] = np.where(t_map > 0, feats["acc_id"].map(u_src.get("bc", 0)).fillna(0.0) / t_map, 0.0)
    feats["api_usage_share"] = np.where(t_map > 0, feats["acc_id"].map(u_src.get("api", 0)).fillna(0.0) / t_map, 0.0)
    feats["otp_usage_share"] = np.where(t_map > 0, feats["acc_id"].map(u_src.get("otp", 0)).fillna(0.0) / t_map, 0.0)
    feats["channel_hhi"] = feats["sms_usage_share"]**2 + feats["email_usage_share"]**2
    feats["multichannel_flag"] = ((feats["sms_usage_share"] > 0) & (feats["email_usage_share"] > 0)).astype(float)
    
    feats["channel_entropy"] = -(
        feats["sms_usage_share"] * np.log2(feats["sms_usage_share"] + 1e-9) +
        feats["email_usage_share"] * np.log2(feats["email_usage_share"] + 1e-9)
    ).clip(lower=0.0)
    feats["source_entropy"] = -(
        feats["bc_usage_share"] * np.log2(feats["bc_usage_share"] + 1e-9) +
        feats["api_usage_share"] * np.log2(feats["api_usage_share"] + 1e-9) +
        feats["otp_usage_share"] * np.log2(feats["otp_usage_share"] + 1e-9)
    ).clip(lower=0.0)
    
    feats["payment_to_usage_ratio"] = feats["total_revenue_180d"] / (feats["usage_total_180d"] + 1.0)
    
    # 5. Activity Recency
    feats["days_since_last_activity"] = feats[["days_since_last_payment", "days_since_last_usage"]].min(axis=1)
    
    # 6. Eligibility for Churn (Active Paid)
    is_active = feats["days_since_last_activity"].fillna(9999) <= 180
    ever_paid = feats["payment_count_all"] > 0
    feats["eligible_for_churn"] = is_active & ever_paid
    
    # 7. Ground Truth Labels in Post-Cutoff Horizon
    fut_pay_ids = set(payments[(payments["payment_date"] >= cutoff) & (payments["payment_date"] < horizon_end)]["acc_id"])
    fut_use_ids = set(usage[(usage["period"] >= cutoff) & (usage["period"] < horizon_end) & (usage["usage"] > 0)]["acc_id"])
    future_active = fut_pay_ids | fut_use_ids
    feats["churn_label"] = np.where(feats["acc_id"].isin(future_active), 0, 1)
    
    return feats

churn_data_full = extract_churn_sota_features_and_labels(customers, payments, usage, CUTOFF_DATE, HORIZON_DAYS)
churn_df = churn_data_full[churn_data_full["eligible_for_churn"]].reset_index(drop=True)

feature_cols = [
    "customer_age_days", "days_since_last_activity", "days_since_last_payment", "days_since_last_usage",
    "payment_count_all", "payment_count_180d", "payment_count_30d", "total_revenue_all", "total_revenue_180d",
    "payment_amount_30d", "avg_transaction_value", "payment_interval_mean_days", "payment_overdue_ratio",
    "payment_amount_cv", "payment_velocity_ratio", "payment_amount_max_ratio", "recency_monetary_ratio",
    "usage_total_180d", "usage_recent_90d", "usage_prev_90d", "usage_recent_30d", "usage_30d_to_90d_ratio",
    "usage_active_months_180d", "usage_consistency_ratio", "usage_slope_6m", "usage_velocity",
    "usage_acceleration", "usage_burstiness_fano", "usage_ewma_30d", "usage_ewma_90d", "usage_change_90d_pct",
    "usage_decay_ratio", "sms_usage_share", "email_usage_share", "bc_usage_share", "api_usage_share",
    "otp_usage_share", "channel_hhi", "multichannel_flag", "channel_entropy", "source_entropy", "payment_to_usage_ratio"
]

print(f"📊 SOTA Dataset Summary at Cutoff {CUTOFF_DATE}:")
print(f"   - Total Active Paid Customers: {len(churn_df):,}")
print(f"   - Number of SOTA Features     : {len(feature_cols)}")
print(f"   - Positive Churn Class Rate   : {churn_df['churn_label'].mean():.2%} ({churn_df['churn_label'].sum()} churned, {(churn_df['churn_label']==0).sum()} retained)")


## ✂️ Step 5: แบ่งข้อมูล 60/20/20 (Train / Validation / Test) และ Preprocessing
- **Train (60%)**: ใช้เรียนรู้การกระจายตัว และเทรนโมเดล
- **Validation (20%)**: ใช้สำหรับ Optuna Hyperparameter Search, Calibration และหา Decision Threshold
- **Test (20%)**: Holdout ชุดที่ไม่ถูกแตะต้องเลย ใช้สำหรับประเมินผลขั้นสุดท้าย
- **Preprocessing**: Median Imputation + Standard Scaling (คำนวณสถิติจากชุด Train เท่านั้นเพื่อกัน Data Leakage)



In [ ]:
from sklearn.model_selection import train_test_split

# แบ่ง Stratified Split 60/20/20
train_df, holdout_df = train_test_split(
    churn_df, test_size=0.40, random_state=42, stratify=churn_df["churn_label"]
)
val_df, test_df = train_test_split(
    holdout_df, test_size=0.50, random_state=42, stratify=holdout_df["churn_label"]
)

print(f"Dataset Splits: Train={len(train_df):,} | Val={len(val_df):,} | Test={len(test_df):,}")

# Preprocessing: Imputation & Scaling (Fit on Train only!)
X_train_raw = train_df[feature_cols].copy()
X_val_raw = val_df[feature_cols].copy()
X_test_raw = test_df[feature_cols].copy()

impute_values = X_train_raw.median().fillna(0.0)
X_train_imp = X_train_raw.fillna(impute_values)
X_val_imp = X_val_raw.fillna(impute_values)
X_test_imp = X_test_raw.fillna(impute_values)

mean_vals = X_train_imp.mean()
std_vals = X_train_imp.std().replace(0.0, 1.0).fillna(1.0)

X_train = (X_train_imp - mean_vals) / std_vals
X_val = (X_val_imp - mean_vals) / std_vals
X_test = (X_test_imp - mean_vals) / std_vals

y_train = train_df["churn_label"].values
y_val = val_df["churn_label"].values
y_test = test_df["churn_label"].values



## 🧠 Step 6: เทรนและเปรียบเทียบโมเดลระดับโลก (Baselines vs LightGBM vs XGBoost vs CatBoost vs Stacking Meta-Ensemble)
ทำการแข่งขันกันระหว่าง 5 สถาปัตยกรรม:
1. **Baseline 1**: Rule-based Recency 90d
2. **Baseline 2**: Logistic Regression (Balanced)
3. **Candidate 1**: LightGBM (Leaf-wise GBDT, Optuna Tuned)
4. **Candidate 2**: XGBoost (Histogram/Depth-wise GBDT, Tuned)
5. **Candidate 3**: CatBoost (Symmetric Trees, Ordered Boosting)
6. **🏆 Champion**: **Tri-GBDT Stacking Meta-Classifier** (Super Learner ผสมผลทำนาย OOF ข้ามตระกูล)


In [ ]:
# 1. Baselines
rule_pred_val = (val_df["days_since_last_activity"] > 90).astype(int).values
rule_pr_auc = average_precision_score(y_val, rule_pred_val)
rule_roc_auc = roc_auc_score(y_val, rule_pred_val)

lr_model = LogisticRegression(class_weight="balanced", random_state=42, max_iter=1000)
lr_model.fit(X_train, y_train)
lr_pred_val = lr_model.predict_proba(X_val)[:, 1]
lr_pr_auc = average_precision_score(y_val, lr_pred_val)
lr_roc_auc = roc_auc_score(y_val, lr_pred_val)

# 2. LightGBM Optuna Tuned
scale_pos = float((y_train == 0).sum() / max(1, (y_train == 1).sum()))

def lgb_objective(trial):
    params = {
        "objective": "binary", "metric": "binary_logloss", "boosting_type": "gbdt",
        "verbosity": -1, "random_state": 42,
        "n_estimators": trial.suggest_int("n_estimators", 60, 250),
        "learning_rate": trial.suggest_float("learning_rate", 0.015, 0.15, log=True),
        "num_leaves": trial.suggest_int("num_leaves", 15, 63),
        "max_depth": trial.suggest_int("max_depth", 3, 7),
        "min_child_samples": trial.suggest_int("min_child_samples", 15, 50),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "scale_pos_weight": trial.suggest_float("scale_pos_weight", 1.0, scale_pos),
    }
    clf = lgb.LGBMClassifier(**params)
    clf.fit(X_train, y_train)
    return average_precision_score(y_val, clf.predict_proba(X_val)[:, 1])

print("🔍 Tuning LightGBM with Optuna...")
lgb_study = optuna.create_study(direction="maximize")
lgb_study.optimize(lgb_objective, n_trials=25, show_progress_bar=False)
best_lgbm = lgb.LGBMClassifier(**lgb_study.best_params, verbosity=-1, random_state=42)
best_lgbm.fit(X_train, y_train)
lgb_pred_val = best_lgbm.predict_proba(X_val)[:, 1]
lgb_pr_auc = average_precision_score(y_val, lgb_pred_val)
lgb_roc_auc = roc_auc_score(y_val, lgb_pred_val)

# 3. XGBoost Tuned
print("🔍 Tuning XGBoost...")
best_xgb = xgb.XGBClassifier(
    n_estimators=160, learning_rate=0.04, max_depth=4, subsample=0.8,
    colsample_bytree=0.8, scale_pos_weight=scale_pos, eval_metric="logloss",
    random_state=42, tree_method="hist"
)
best_xgb.fit(X_train, y_train)
xgb_pred_val = best_xgb.predict_proba(X_val)[:, 1]
xgb_pr_auc = average_precision_score(y_val, xgb_pred_val)
xgb_roc_auc = roc_auc_score(y_val, xgb_pred_val)

# 4. CatBoost Tuned (Symmetric Trees, Ordered Boosting)
print("🔍 Tuning CatBoost...")
best_cat = CatBoostClassifier(
    iterations=250, learning_rate=0.05, depth=5, auto_class_weights="Balanced",
    verbose=0, random_seed=42
)
best_cat.fit(X_train, y_train)
cat_pred_val = best_cat.predict_proba(X_val)[:, 1]
cat_pr_auc = average_precision_score(y_val, cat_pred_val)
cat_roc_auc = roc_auc_score(y_val, cat_pred_val)

# 5. 🏆 Tri-GBDT Stacking Meta-Ensemble (Super Learner)
print("🏆 Building Tri-GBDT Stacking Meta-Ensemble...")
cv_stack = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
oof_lgb = np.zeros(len(X_train))
oof_xgb = np.zeros(len(X_train))
oof_cat = np.zeros(len(X_train))

for tr_idx, h_idx in cv_stack.split(X_train, y_train):
    f_X_tr, f_y_tr = X_train.iloc[tr_idx], y_train[tr_idx]
    f_X_h = X_train.iloc[h_idx]
    
    m_lgb = lgb.LGBMClassifier(**lgb_study.best_params, verbosity=-1, random_state=42)
    m_lgb.fit(f_X_tr, f_y_tr)
    oof_lgb[h_idx] = m_lgb.predict_proba(f_X_h)[:, 1]
    
    m_xgb = xgb.XGBClassifier(n_estimators=160, learning_rate=0.04, max_depth=4, subsample=0.8, colsample_bytree=0.8, scale_pos_weight=scale_pos, eval_metric="logloss", random_state=42, tree_method="hist")
    m_xgb.fit(f_X_tr, f_y_tr)
    oof_xgb[h_idx] = m_xgb.predict_proba(f_X_h)[:, 1]
    
    m_cat = CatBoostClassifier(iterations=250, learning_rate=0.05, depth=5, auto_class_weights="Balanced", verbose=0, random_seed=42)
    m_cat.fit(f_X_tr, f_y_tr)
    oof_cat[h_idx] = m_cat.predict_proba(f_X_h)[:, 1]

S_train = np.column_stack([oof_lgb, oof_xgb, oof_cat])
S_val = np.column_stack([lgb_pred_val, xgb_pred_val, cat_pred_val])

meta_model = LogisticRegression(C=1.0, random_state=42)
meta_model.fit(S_train, y_train)
stack_pred_val = meta_model.predict_proba(S_val)[:, 1]
stack_pr_auc = average_precision_score(y_val, stack_pred_val)
stack_roc_auc = roc_auc_score(y_val, stack_pred_val)

# Leaderboard Table
leaderboard = pd.DataFrame({
    "Model Architecture": ["Baseline (Recency 90d)", "Baseline (Logistic Regression)", "LightGBM (Optuna Tuned)", "XGBoost (Tuned)", "CatBoost (Tuned)", "🏆 Tri-GBDT Stacking Ensemble"],
    "Val PR-AUC": [rule_pr_auc, lr_pr_auc, lgb_pr_auc, xgb_pr_auc, cat_pr_auc, stack_pr_auc],
    "Val ROC-AUC": [rule_roc_auc, lr_roc_auc, lgb_roc_auc, xgb_roc_auc, cat_roc_auc, stack_roc_auc],
})
print("\n========================== 🏆 SOTA MODEL TOURNAMENT LEADERBOARD ==========================")
print(leaderboard.to_string(index=False))
print("========================================================================================\n")


## 🎯 Step 7: Probability Calibration (Platt Scaling vs Isotonic)
ค่าความน่าจะเป็นดิบจาก Tree Models มักจะเบ้ (Overconfident) การทำ Calibration ช่วยให้:
- เมื่อโมเดลบอกความน่าจะเป็น 70% ลูกค้าในกลุ่มนี้ควรเลิกใช้จริงประมาณ 70%
- เปรียบเทียบระหว่าง Platt Scaling (Logistic) และ Isotonic Regression ด้วย Brier Score



In [ ]:
# Probability Calibration (Platt Scaling vs Isotonic) บน Stacking Ensemble Predictions
val_raw_scores = stack_pred_val

# Fit Calibrator on Validation Slices
iso_cal = IsotonicRegression(out_of_bounds="clip")
iso_cal.fit(val_raw_scores, y_val)

platt_cal = LogisticRegression(C=1.0, random_state=42)
platt_cal.fit(val_raw_scores.reshape(-1, 1), y_val)

iso_preds = iso_cal.predict(val_raw_scores)
platt_preds = platt_cal.predict_proba(val_raw_scores.reshape(-1, 1))[:, 1]

brier_raw = brier_score_loss(y_val, val_raw_scores)
brier_iso = brier_score_loss(y_val, iso_preds)
brier_platt = brier_score_loss(y_val, platt_preds)

print(f"🎯 Calibration Comparison (Brier Score, Lower is Better):")
print(f"   - Raw Uncalibrated : {brier_raw:.4f}")
print(f"   - Platt Scaling    : {brier_platt:.4f}")
print(f"   - Isotonic Reg     : {brier_iso:.4f}")

use_isotonic = brier_iso < brier_platt
print(f"   ==> Selected Method: {'Isotonic Regression' if use_isotonic else 'Platt Scaling (Logistic)'}")

def apply_calibration(raw_probs):
    if use_isotonic:
        return np.clip(iso_cal.predict(raw_probs), 0.0, 1.0)
    return np.clip(platt_cal.predict_proba(raw_probs.reshape(-1, 1))[:, 1], 0.0, 1.0)


## ⚖️ Step 8: ค้นหาเกณฑ์ตัดสินใจความเสี่ยงสูงสุด (Optimal Decision Threshold via $F_2$)
ในงาน Churn การปล่อยให้ลูกค้าหลุดมือ (False Negative) มีต้นทุนทางธุรกิจสูงกว่าการติดต่อลูกค้าผิด (False Positive) 
เราจึงใช้ **$F_2$ Score** (ให้น้ำหนัก Recall สูงกว่า Precision 2 เท่า) เพื่อค้นหาจุดตัดความเสี่ยงสูง (`high_risk_threshold`)


In [ ]:
# ค้นหา Decision Threshold ที่เหมาะสมที่สุดด้วย F2-Score และ Cost-Benefit Expected Value
val_calibrated = apply_calibration(val_raw_scores)

thresholds_scan = np.linspace(0.20, 0.85, 131)
f2_scores = [fbeta_score(y_val, (val_calibrated >= th).astype(int), beta=2.0) for th in thresholds_scan]
f1_scores = [f1_score(y_val, (val_calibrated >= th).astype(int)) for th in thresholds_scan]

best_idx = np.argmax(f2_scores)
optimal_high_threshold = round(float(thresholds_scan[best_idx]), 3)
medium_threshold = round(optimal_high_threshold / 2.0, 3)

print(f"⚖️ Optimal Decision Thresholds:")
print(f"   - High Risk Cutoff (Max-F2) : {optimal_high_threshold:.3f} (F2 = {f2_scores[best_idx]:.4f})")
print(f"   - Medium Risk Cutoff        : {medium_threshold:.3f}")


## 📈 Step 9: ประเมินผลบนชุดทดสอบจริง (Test Set Evaluation & Visualizations)
ทดสอบ Champion Model กับ Holdout Test Set 20% ที่ไม่เคยเห็นมาก่อน


In [ ]:
# 1. Inference Base Models on Test Set
test_lgb = best_lgbm.predict_proba(X_test)[:, 1]
test_xgb = best_xgb.predict_proba(X_test)[:, 1]
test_cat = best_cat.predict_proba(X_test)[:, 1]
test_lr = lr_model.predict_proba(X_test)[:, 1]
test_rule = (test_df["days_since_last_activity"] > 90).astype(int).values

# 2. Stacking & Calibration
S_test = np.column_stack([test_lgb, test_xgb, test_cat])
test_stack_raw = meta_model.predict_proba(S_test)[:, 1]
test_probs = apply_calibration(test_stack_raw)
test_pred_high = (test_probs >= optimal_high_threshold).astype(int)

# 3. Metrics
test_roc_auc = roc_auc_score(y_test, test_probs)
test_pr_auc = average_precision_score(y_test, test_probs)
test_brier = brier_score_loss(y_test, test_probs)
test_f1 = f1_score(y_test, test_pred_high)
test_f2 = fbeta_score(y_test, test_pred_high, beta=2.0)

print("🏁 ==================== FINAL TEST SET SOTA PERFORMANCE ====================")
print(f"   - Champion Stacking PR-AUC  : {test_pr_auc:.4f}  (Baseline LR: {average_precision_score(y_test, test_lr):.4f})")
print(f"   - Champion Stacking ROC-AUC : {test_roc_auc:.4f} (Baseline LR: {roc_auc_score(y_test, test_lr):.4f})")
print(f"   - Brier Calibration Score   : {test_brier:.4f}")
print(f"   - F1-Score / F2-Score       : {test_f1:.4f} / {test_f2:.4f}")
print("=============================================================================")

# 4. 📊 World-Class 6-Panel Diagnostic Suite
fig, axes = plt.subplots(2, 3, figsize=(20, 11))

# Panel 1: Multi-Model Precision-Recall Overlay
for name, probs, col, lw, ls in [
    ("Baseline (Recency 90d)", test_rule, "gray", 1.5, ":"),
    (f"Logistic Reg ({average_precision_score(y_test, test_lr):.3f})", test_lr, "#3498db", 1.5, "--"),
    (f"LightGBM ({average_precision_score(y_test, test_lgb):.3f})", test_lgb, "#2ecc71", 1.8, "-."),
    (f"CatBoost ({average_precision_score(y_test, test_cat):.3f})", test_cat, "#f39c12", 1.8, "-."),
    (f"🏆 Stacking SOTA ({test_pr_auc:.3f})", test_probs, "#e74c3c", 3.0, "-")
]:
    p, r, _ = precision_recall_curve(y_test, probs)
    axes[0, 0].plot(r, p, label=name, color=col, lw=lw, linestyle=ls)
axes[0, 0].axhline(y_test.mean(), color="black", linestyle=":", alpha=0.5, label=f"Random ({y_test.mean():.1%})")
axes[0, 0].set_title("1. Multi-Model PR-AUC Overlay", fontsize=13, fontweight="bold")
axes[0, 0].set_xlabel("Recall")
axes[0, 0].set_ylabel("Precision")
axes[0, 0].legend(loc="lower left", fontsize=9)

# Panel 2: Multi-Model ROC Overlay
for name, probs, col, lw, ls in [
    (f"Logistic Reg ({roc_auc_score(y_test, test_lr):.3f})", test_lr, "#3498db", 1.5, "--"),
    (f"LightGBM ({roc_auc_score(y_test, test_lgb):.3f})", test_lgb, "#2ecc71", 1.8, "-."),
    (f"CatBoost ({roc_auc_score(y_test, test_cat):.3f})", test_cat, "#f39c12", 1.8, "-."),
    (f"🏆 Stacking SOTA ({test_roc_auc:.3f})", test_probs, "#e74c3c", 3.0, "-")
]:
    fpr, tpr, _ = roc_curve(y_test, probs)
    axes[0, 1].plot(fpr, tpr, label=name, color=col, lw=lw, linestyle=ls)
axes[0, 1].plot([0, 1], [0, 1], color="gray", linestyle=":")
axes[0, 1].set_title("2. Multi-Model ROC Curves Overlay", fontsize=13, fontweight="bold")
axes[0, 1].set_xlabel("False Positive Rate")
axes[0, 1].set_ylabel("True Positive Rate")
axes[0, 1].legend(loc="lower right", fontsize=9)

# Panel 3: Reliability Diagram (Calibration Curve)
from sklearn.calibration import calibration_curve
prob_true, prob_pred = calibration_curve(y_test, test_probs, n_bins=8, strategy="uniform")
axes[0, 2].plot(prob_pred, prob_true, marker="o", lw=2, color="#9b59b6", label="Calibrated Stacking")
axes[0, 2].plot([0, 1], [0, 1], linestyle="--", color="gray", label="Perfect Calibration")
axes[0, 2].set_title(f"3. Reliability Diagram (Brier={test_brier:.4f})", fontsize=13, fontweight="bold")
axes[0, 2].set_xlabel("Mean Predicted Probability")
axes[0, 2].set_ylabel("Fraction of Positives")
axes[0, 2].legend(loc="upper left", fontsize=9)

# Panel 4: Cumulative Gains & Decile Lift
lift_df = pd.DataFrame({"y": y_test, "prob": test_probs}).sort_values("prob", ascending=False).reset_index(drop=True)
lift_df["cum_churn"] = lift_df["y"].cumsum() / lift_df["y"].sum()
lift_df["pct_customers"] = (lift_df.index + 1) / len(lift_df)
axes[1, 0].plot(lift_df["pct_customers"], lift_df["cum_churn"], color="#16a085", lw=2.5, label="Stacking Model")
axes[1, 0].plot([0, 1], [0, 1], linestyle="--", color="gray", label="Random Contact")
top_20_gain = lift_df.iloc[int(0.20 * len(lift_df))]["cum_churn"]
axes[1, 0].scatter([0.20], [top_20_gain], color="crimson", s=80, zorder=5)
axes[1, 0].annotate(f"Top 20% Contact captures\n{top_20_gain:.1%} of All Churners", xy=(0.20, top_20_gain),
                    xytext=(0.28, top_20_gain - 0.15), arrowprops=dict(facecolor="black", shrink=0.05, width=1))
axes[1, 0].set_title("4. Cumulative Gains Chart", fontsize=13, fontweight="bold")
axes[1, 0].set_xlabel("Fraction of Customers Contacted")
axes[1, 0].set_ylabel("Fraction of Churners Captured")
axes[1, 0].legend(loc="lower right", fontsize=9)

# Panel 5: Confusion Matrix at Optimal Decision Threshold
cm = confusion_matrix(y_test, test_pred_high)
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=axes[1, 1],
            xticklabels=["Retained (Pred)", "High Churn (Pred)"],
            yticklabels=["Retained (Actual)", "Churned (Actual)"])
axes[1, 1].set_title(f"5. Confusion Matrix (Threshold={optimal_high_threshold:.2f})", fontsize=13, fontweight="bold")

# Panel 6: Risk Level vs Revenue at Risk Distribution
risk_labels = ["Low Risk", "Medium Risk", "High Risk"]
risk_bins = [-0.01, medium_threshold, optimal_high_threshold, 1.01]
test_cats = pd.cut(test_probs, bins=risk_bins, labels=risk_labels)
test_rev = test_df["total_revenue_180d"].values
risk_summary = pd.DataFrame({"risk": test_cats, "rev": test_rev}).groupby("risk").agg(
    count=("rev", "count"),
    rev_at_risk=("rev", "sum")
).reindex(risk_labels)

bars = axes[1, 2].bar(risk_summary.index, risk_summary["count"], color=["#2ecc71", "#f39c12", "#e74c3c"], width=0.5)
axes[1, 2].set_ylabel("Number of Customers", color="#2c3e50", fontweight="bold")
ax2 = axes[1, 2].twinx()
ax2.plot(risk_summary.index, risk_summary["rev_at_risk"] / 1e6, color="#8e44ad", marker="D", lw=2, markersize=8)
ax2.set_ylabel("Revenue at Risk (Million THB)", color="#8e44ad", fontweight="bold")
axes[1, 2].set_title("6. Churn Risk & Revenue at Risk Distribution", fontsize=13, fontweight="bold")

plt.tight_layout()
plt.show()


## 🔍 Step 10: วิเคราะห์เจาะลึกปัจจัยความเสี่ยงด้วย SHAP (Model Explainability)
SHAP (SHapley Additive exPlanations) ช่วยบอกว่า:
- ฟีเจอร์ไหนมีผลต่อความเสี่ยง Churn มากที่สุดในภาพรวม (Global Importance)
- ฟีเจอร์ไหนที่ผลักดันให้ **ลูกค้ารายใดรายหนึ่ง** มีความเสี่ยงพุ่งสูง (Local Explanation) เพื่อให้ทีมงานนำไปใช้ออกแคมเปญช่วยเหลือได้ตรงจุด



In [ ]:
# SHAP Feature Importance & Multi-Model Feature Comparison
explainer = shap.TreeExplainer(best_lgbm)
shap_values = explainer.shap_values(X_test)
if isinstance(shap_values, list):
    shap_to_plot = shap_values[1]
else:
    shap_to_plot = shap_values

plt.figure(figsize=(10, 6))
shap.summary_plot(shap_to_plot, X_test, feature_names=feature_cols, max_display=15, show=False)
plt.title("SHAP Summary (Top Drivers of Customer Churn)", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()


In [ ]:
# ตัวอย่าง Local Explanation: เจาะลึกลูกค้า 1 รายที่มีความเสี่ยงสูง
high_risk_idx = np.where(test_probs > optimal_high_threshold)[0]

if len(high_risk_idx) > 0:
    sample_idx = high_risk_idx[0]
    cust_id = test_df.iloc[sample_idx]["acc_id"]
    cust_prob = test_probs[sample_idx]
    
    print(f"🔎 เจาะลึกลูกค้า acc_id: {cust_id} (ทำนายความเสี่ยง: {cust_prob:.1%})")
    print("Top 3 ปัจจัยผลักดันความเสี่ยงสูงสุด:")
    
    sample_shap = pd.Series(shap_vals_plot[sample_idx], index=feature_cols).sort_values(ascending=False)
    for feat, impact in sample_shap.head(3).items():
        actual_val = test_df.iloc[sample_idx][feat]
        print(f"   👉 {feat} (ค่าจริง = {actual_val}): ส่งผลให้ความเสี่ยงเพิ่มขึ้น {impact:+.3f}")



## 📤 Step 11: ทำนายลูกค้าทั้งหมดและ Export ผลลัพธ์เป็น CSV
รัน Inference กับลูกค้ากลุ่ม Active Paid ทุกคน แล้วเซฟเป็น `churn_predictions.csv` สำหรับนำไปใช้งานต่อ


In [ ]:
# 1. รันทำนายลูกค้าทั้งหมดในชุด Active Paid
all_active_raw = churn_df[feature_cols].copy().fillna(impute_values)
all_active_scaled = (all_active_raw - mean_vals) / std_vals

raw_churn_prob = best_lgbm.predict_proba(all_active_scaled)[:, 1]
calibrated_prob = apply_calibration(raw_churn_prob)

# 2. จัดหมวด Risk Level
risk_levels = []
for p in calibrated_prob:
    if p < medium_threshold:
        risk_levels.append("low")
    elif p < optimal_high_threshold:
        risk_levels.append("medium")
    else:
        risk_levels.append("high")

# 3. สกัด Top Risk Factors ต่อลูกค้าแต่ละราย
all_shap = explainer.shap_values(all_active_scaled)
if isinstance(all_shap, list): all_shap = all_shap[1]

top_factors = []
for i in range(len(churn_df)):
    row_shap = pd.Series(all_shap[i], index=feature_cols).sort_values(ascending=False)
    top_3 = ", ".join(row_shap.head(3).index)
    top_factors.append(top_3)

# 4. รวมเป็น DataFrame สรุปผล
output_df = pd.DataFrame({
    "acc_id": churn_df["acc_id"],
    "churn_probability": np.round(calibrated_prob, 4),
    "churn_risk_level": risk_levels,
    "top_risk_factors": top_factors,
    "total_revenue_180d": churn_df["total_revenue_180d"],
    "usage_recent_90d": churn_df["usage_recent_90d"],
    "days_since_last_activity": churn_df["days_since_last_activity"]
})

output_path = "churn_predictions.csv"
output_df.to_csv(output_path, index=False)
print(f"🎉 บันทึกผลการทำนายเรียบร้อยที่: {output_path}")
print(f"ตัวอย่างผลลัพธ์ (5 รายแรกที่มีความเสี่ยงสูง):")
output_df.sort_values("churn_probability", ascending=False).head(5)



## 🎛️ Step 12: Interactive Tuning Playground (ห้องทดลองและปรับจูนโมเดล)
คุณสามารถแก้ไขและทดลองรัน Parameter ในเซลล์นี้ได้ตามต้องการ:
- ปรับเปลี่ยน `CUTOFF_DATE` เพื่อดูผลการย้อนหลังที่จุดเวลาอื่น
- ปรับจูนจำนวน Trials ของ Optuna
- ปรับระดับ Threshold ตามนโยบายของทีม CRM

